In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQA
from dotenv import load_dotenv
import os
from langchain_core.prompts import PromptTemplate
from langchain.output_parsers import PydanticOutputParser, OutputFixingParser
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from pydantic import BaseModel, Field

# Load env
load_dotenv()
huggingFaceToken = os.getenv("HUGGINGFACEHUB_ACCESS_TOKEN")

# ==========================
# Define output schema
# ==========================
class Answer(BaseModel):
    answer: str = Field(..., description="answer to the question based on the document content")

# ==========================
# Setup LLM
# ==========================
llm_raw = HuggingFaceEndpoint(
    repo_id="google/gemma-2-2b-it",
    task="text-generation",
    huggingfacehub_api_token=huggingFaceToken,
    temperature=0.0
)
model = ChatHuggingFace(llm=llm_raw)

# Pydantic parser
parser = PydanticOutputParser(pydantic_object=Answer)
answer_parser = OutputFixingParser.from_llm(parser=parser, llm=model)

# ==========================
# Load and process PDF
# ==========================
loader = PyPDFLoader("24869-pdf.pdf")
docs = loader.load()

# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
splits = text_splitter.split_documents(docs)

# ==========================
# Create vector store
# ==========================
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(splits, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# ==========================
# Create QA prompt
# ==========================
qa_prompt = PromptTemplate(
    template="""Use the following pieces of context to answer the question at the end. 
    If you don't know the answer based on the context, just say that you don't know, 
    don't try to make up an answer.

    Context:
    {context}

    Question: {question}
    
    Answer: Provide a clear and concise answer based on the context above.
    
    {format_instructions}""",
    input_variables=["context", "question"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

# ==========================
# Create QA chain
# ==========================
qa_chain = RetrievalQA.from_chain_type(
    llm=model,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": qa_prompt},
    return_source_documents=True
)

# Alternative approach using LCEL (LangChain Expression Language)
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | qa_prompt
    | model
    | answer_parser
)

# ==========================
# Function to ask questions
# ==========================
def ask_question(question: str, use_rag_chain: bool = True):
    """Ask a question about the PDF content"""
    if use_rag_chain:
        try:
            result = rag_chain.invoke(question)
            return result.answer
        except Exception as e:
            print(f"Error with RAG chain: {e}")
            # Fallback to basic chain
            result = qa_chain({"query": question})
            return result["result"]
    else:
        result = qa_chain({"query": question})
        return result["result"], result["source_documents"]

# ==========================
# Example usage
# ==========================
if __name__ == "__main__":
    # Ask questions about your PDF
    questions = [
        "What is the main topic of this document?",
        "What are the key points discussed?",
        "Can you summarize the first chapter?"
    ]
    
    for question in questions:
        print(f"\nQuestion: {question}")
        answer = ask_question(question)
        print(f"Answer: {answer}")
        print("-" * 50)
    
    # Interactive mode
    while True:
        user_question = input("\nAsk a question about the PDF (or 'quit' to exit): ")
        if user_question.lower() == 'quit':
            break
        
        try:
            answer = ask_question(user_question)
            print(f"Answer: {answer}")
        except Exception as e:
            print(f"Error: {e}")

KeyboardInterrupt: 